# Local Dataset Smoke Test

This notebook checks the Python package against the bundled CFG dataset without running LAMMPS.

In [ ]:
from pathlib import Path
import os


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "interface_analyzer" / "reproducibility").exists():
            return path
    raise RuntimeError("Could not find repository root containing interface_analyzer/reproducibility")


PROJECT_ROOT = find_repo_root()
REPRO_DIR = PROJECT_ROOT / "interface_analyzer" / "reproducibility"
DATASET_DIR = REPRO_DIR / "dataset"
LOCAL_OUTPUT_DIR = REPRO_DIR / "_local_outputs"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full manuscript-scale post-processing files are intentionally not bundled.
# Set INTERFACE_ANALYZER_DATA to the directory containing those generated files.
FULL_DATA_ROOT = Path(os.environ.get("INTERFACE_ANALYZER_DATA", LOCAL_OUTPUT_DIR)).expanduser()
FULL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# For quick local CFG tests this defaults to the bundled sample dataset.
CFG_DIR = Path(os.environ.get("INTERFACE_ANALYZER_CFG_DIR", DATASET_DIR)).expanduser()

print("Project root:", PROJECT_ROOT)
print("Bundled CFG dataset:", DATASET_DIR)
print("Analysis data root:", FULL_DATA_ROOT)
print("CFG input dir:", CFG_DIR)


In [ ]:
from pathlib import Path
import pickle

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(p for p in candidates if (p / "interface_analyzer" / "reproducibility" / "dataset").exists())
DATASET_DIR = PROJECT_ROOT / "interface_analyzer" / "reproducibility" / "dataset"
OUTPUT_DIR = PROJECT_ROOT / "interface_analyzer" / "reproducibility" / "_local_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cfg_files = sorted(DATASET_DIR.glob("cfg.Al_100_010.*"), key=lambda p: int(p.name.split(".")[-1]))
cfg_path = cfg_files[0]

print("Project root:", PROJECT_ROOT)
print("Dataset dir:", DATASET_DIR)
print("CFG files:", len(cfg_files))
print("Using:", cfg_path.name)

In [ ]:
from interface_analyzer import Orientation_analysis, analyze_cfm

result = Orientation_analysis(
    str(cfg_path),
    lattice_constant=4.134,
    miller_x=[1, 0, 0],
    miller_y=[0, 1, 0],
    miller_z=[0, 0, 1],
    a_grid=2.5,
    d=6.0,
    n=5,
    solid_value=1,
    liquid_value=2,
)

frame_id = int(cfg_path.name.split(".")[-1])
pickle_path = OUTPUT_DIR / "local_smoke_orientation.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump({frame_id: result}, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved:", pickle_path)
print("Result keys:", sorted(result))
print("Interface bins:", len(result["x"]))

In [ ]:
cfm = analyze_cfm(pickle_path, T=930, a=4.134, show_plot=False, pchipres=256)

print("Snapshots:", cfm["snapshots"])
print("k points:", len(cfm["k"]))
print("Lx:", round(cfm["Lx"], 3))
print("Smoke test complete")